# SWIGGY SALES ANALYSIS

## Import Libraries

In [1]:
import pandas as pd  # to cleaning, aggration in data
import numpy as np    # to do complex mathematical operations
import matplotlib.pyplot as plt # to create different charts (basic level charts)
import seaborn as sns    # advance version of matplotlib
import plotly.express as px   # filtering charts 

### Import Data

In [ ]:
df = pd.read_excel('swiggy_data.xlsx')

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

### MetaData

In [ ]:
print("Number of Rows:", df.shape[0])   # Number of rows

In [ ]:
print("No of Fields:", df.shape[1])   # Number of columns

In [ ]:
df.info     # sampel data, identifying data

### Data Types

In [ ]:
df.dtypes     # checking the all datatypes

In [ ]:
df.describe()   # all information

# KPI's

### Total Sales

In [ ]:
total_sales = df["Price (INR)"].sum()
print("Total Sales (INR):", round(total_sales,2))

### Average Rating

In [ ]:
average_rating = df["Rating"].mean()
print("Average Rating:", round(average_rating,1))

### Average Order Value

In [ ]:
average_order_value = df["Price (INR)"].mean()
print("Average Order Value (INR):", round(average_order_value,2))

### Ratings Count

In [ ]:
ratings_count = df["Rating Count"].sum()
print("Rating Count:", round(ratings_count,2))

### Total Orders

In [ ]:
total_orders = len(df)
print(total_orders)

# Charts Design


## Monthly Sales Trend


In [ ]:
# df["Order Date"] = pd.to_datetime = (df["Order Date"])

df["YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)

monthly_revenue = df.groupby("YearMonth")["Price (INR)"].sum().reset_index()

plt.figure()
plt.plot(monthly_revenue["YearMonth"], monthly_revenue["Price (INR)"])
plt.xticks(rotation=45)
plt.xlabel("Month")
plt.ylabel("Total Sales (INR)")
plt.title("Monthly Sales Trend")
plt.tight_layout()
plt.show()


## Daily Sales Trend


In [ ]:
df.columns = df.columns.str.strip()

df["Order Date"] = pd.to_datetime(df["Order Date"])

df["DayName"] = df["Order Date"].dt.day_name()

daily_revenue = (
    df.groupby("DayName")["Price (INR)"]
    .sum()
    .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
)

plt.figure(figsize=(10,5))
plt.bar(daily_revenue.index, daily_revenue.values)
plt.style.use("ggplot")
plt.title("Daily Revenue Trend (Mon-Sun)")
plt.xlabel("Day")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=30)

plt.style.use("fivethirtyeight")

bars = plt.bar(daily_revenue.index, daily_revenue.values, color=["red","orange","yellow","green","blue","purple","pink"])

# plt.bar(daily_revenue.index, daily_revenue.values, color=colors)


for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, int(yval),
             ha='center', va='bottom')
    
plt.show()

## Total Sales by Food Type (Veg vs Non-Veg)

In [ ]:
non_veg_keywords = [
    "chicken","egg","fish","mutton",
    "prawn","biryani","kabab","kebab",
    "non-veg","non veg"
]

df["Food Category"] = np.where(
    df["Dish Name"].str.lower().str.contains("|".join(non_veg_keywords), na = False),
    "Non-Veg",
    "Veg"
)

In [ ]:
food_revenue = (
    df.groupby("Food Category")["Price (INR)"]
    .sum()
    .reset_index()
)

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
import plotly.express as px

food_revenue = df.groupby("Food Category")["Price (INR)"].sum().reset_index()

fig = px.pie(
    food_revenue,
    values="Price (INR)",
    names="Food Category",
    hole=0.5,
    title="Revenue Contribution: Veg vs Non-Veg",
)

fig.update_traces(
    textinfo="percent+label",
    pull=[0.05, 0]
)

fig.update_layout(
    height=500,
    margin=dict(t=60,b=40,l=40,r=40)
)


fig.update_traces(
    textinfo="percent+label",
    textfont_size=14,
    marker=dict(colors=["#2ECC71","#E74C3C"]),
    pull=[0.05,0]
)

fig.show()

## Total Sales by State

In [ ]:
fig = px.bar(
    df.groupby("State", as_index=False)["Price (INR)"].sum()
    .sort_values("Price (INR)", ascending=False),
    x = "Price (INR)",
    y = "State",
    orientation = "h",
    title = "Revenue by State (INR)"
)

fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
fig.show()

## Quarterly Performance Summary

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order Date"])
df["Quarter"] = df["Order_Date"].dt.to_period("Q").astype(str)

quarterly_summary = (
    df.groupby("Quarter", as_index=False)
    .agg(
        Total_Sales=("Price (INR)", "sum"),
        Avg_Rating=("Rating", "mean"),
        Total_Orders=("Order_Date", "count")
    )
    .sort_values("Quarter")
)

quarterly_summary["Total_Sales"] = quarterly_summary["Total_Sales"].round(0)
quarterly_summary["Avg_Rating"] = quarterly_summary["Avg_Rating"].round(2)

quarterly_summary

## Top 5 Cities by Sales

In [ ]:
top_5_cities = (
    df.groupby("City")["Price (INR)"]
    .sum()
    .nlargest(5)
    .sort_values()
    .reset_index()
)

fig = px.bar(
    top_5_cities,
    x="Price (INR)",
    y="City",
    orientation="h",
    title="Top 5 Cities by Sales (INR)",
    color_discrete_sequence=["red"]
)

fig.show()

## Weekly Trend Analysis

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order Date"])

df["Week"] = df["Order_Date"].dt.to_period("W").astype(str)

weekly_sales = (
    df.groupby("Week", as_index=False)["Price (INR)"]
    .sum()
)

fig = px.line(
    weekly_sales,
    x="Week",
    y="Price (INR)",
    title="Weekly Sales Trend",
    markers=True
)

fig.update_layout(
    xaxis_title="Week",
    yaxis_title="Revenue (INR)",
    height=500
)

fig.show()